# StoreDNA — Pipeline stages 

Data flows **top to bottom**. This notebook implements **Stage 3** (highlighted).

```
│  1. Source Data          │
│  2. Curate Data          │
│  3. AI Enrichment  ◀── WE ARE HERE │
│  4. Modality Vectors     │
│  5. StoreDNA Builder     │
│  6. Vector Index         │
│  7. Business Output      │
```

| Stage | Name | Status |
|-------|------|--------|
| 1 | Source Data | Complete |
| 2 | Curate Data | Complete |
| **3** | **AI Enrichment** | **Current** |
| 4 | Modality Vectors | Next |
| 5 | StoreDNA Builder | Next |

# Retail Store DNA Builder — Stage 3: AI Enrichment

## What is Stage 3?

Stage 3 turns **clean curated text** (from Stage 2) into **machine-readable AI signals**:

1. **Embeddings** — dense vectors from Azure OpenAI (`text-embedding-3-large`) for each review, news item, report, product, and ops profile.
2. **GPT enrichment** — short store-level summaries and theme tags per modality using our chat deployment (`gpt4-8k` from `.env`).

This is the bridge between **tabular curated data** and **Stage 4 modality vectors** (one vector pool per signal type per store).

---

## What will be done here?

| Input (Stage 2) | AI action | Output (Stage 3) |
|-----------------|-----------|------------------|
| `stg_reviews.csv` | Embed + upload to Azure AI Search | `reviews_enriched.csv` |
| `stg_news.csv` | Embed + upload to Azure AI Search | `news_enriched.csv` |
| `stg_reports.csv` | Embed + upload to Azure AI Search | `reports_enriched.csv` |
| `stg_products.csv` | Embed + upload to Azure AI Search | `products_enriched.csv` |
| `fact_operations_weekly.csv` | Embed each store × week row + upload | `ops_weekly_enriched.csv` |
| `dim_store.csv` | Join retailer name for GPT context | Used in prompts |

All **embedding vectors** are stored in **Azure AI Search** (`AZURE_SEARCH_INDEX_NAME`).
Enriched CSVs (metadata + GPT summaries) go to: **`data/USA_100_Stores/enriched/`**

---

## Procedure

### Step 1 — Load configuration from `.env`

| Variable | Purpose |
|----------|---------|
| `AZURE_OPENAI_API_KEY` | Authentication |
| `AZURE_OPENAI_ENDPOINT` | Azure resource URL |
| `AZURE_OPENAI_API_VERSION` | API version |
| `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` | Embedding model deployment (`text-embedding-3-large`) |
| `AZURE_OPENAI_EMBEDDING_DIMENSIONS` | Vector size (3072 for `text-embedding-3-large`) |
| `AZURE_OPENAI_DEPLOYMENT` | GPT chat deployment for summaries (`gpt4-8k`) |
| `AZURE_SEARCH_ENDPOINT` | Azure AI Search service URL |
| `AZURE_SEARCH_API_KEY` | Search admin/query key |
| `AZURE_SEARCH_INDEX_NAME` | Index for row-level embeddings (default `store-dna-embeddings`) |
| `AZURE_SEARCH_UPLOAD_BATCH_SIZE` | Documents per upload batch (default 500) |

### Step 2 — Load curated tables

Read CSVs from `data/USA_100_Stores/curated/` produced by Stage 2 notebook.

### Step 3 — Build `embed_text` per row

Concatenate the most informative fields into one string per row, e.g.:

- **Review:** `store_id | source | sentiment | review_text`
- **News:** `store_id | city | state | published_date | headline | summary | demand_impact`
- **Report:** `report_id | store_id | report_date | report_type | severity | issue_category | status | reported_by | description`
- **Product:** all curated SKU fields — `sku_id`, `store_id`, `category`, `product_name`, `description`, `attributes`, `brand_type`, `price_tier`, `unit_price`, `pack_size`, `aisle_zone`, `facings`, `weekly_units_sold`, `margin_pct`, `in_stock`, `vendor`
- **Ops weekly:** `store_id | week_end_date | labor_hours | shrink_pct | oos_rate | fulfillment_rate | customer_complaints` (one string per week row)

### Step 4 — Call Azure OpenAI Embeddings API

- Batch texts (default 50 per API call) to `embeddings.create()`
- Upload each row's vector + `content` to **Azure AI Search**
- Record `embedding_model` and `embedding_dim` on enriched CSVs (vectors are not saved locally)

### Step 5 — GPT store-level enrichment (optional)

For each **store × modality** (reviews, news, reports, products):

- Sample up to 12 text snippets for that store
- Ask GPT for JSON: `summary`, `themes`, `sentiment_overall`
- Merge back to row-level enriched tables + `store_modality_summaries.csv`

Set `RUN_GPT_ENRICHMENT = False` to skip GPT (embeddings only — faster/cheaper).

### Step 6 — Write enriched CSVs

GPT summaries are written to `store_modality_summaries.csv`. 

---



## 1. Setup

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_enrichment import (
    EnrichmentConfig,
    add_embed_text,
    embed_texts,
    load_curated_tables,
    run_enrichment,
)
from src.store_dna_search import get_index_document_count

load_dotenv(PROJECT_ROOT / ".env")

CURATED_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "curated"
ENRICHED_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "enriched"

print("Project root:", PROJECT_ROOT)
print("Curated input:", CURATED_DIR)
print("Enriched output:", ENRICHED_DIR)

Project root: d:\AICOE\Retail-StoreDNA
Curated input: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\curated
Enriched output: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\enriched


## 2. Configuration

Adjust these knobs before running on all 100 stores.

In [7]:
# --- Run controls ---
MAX_STORES = None         # None = all 100 stores; use 5 for a quick test
RUN_GPT_ENRICHMENT = False # False = embeddings only (no GPT summaries)
EMBED_BATCH_SIZE = 50   # texts per embedding API call

config = EnrichmentConfig.from_env(
    PROJECT_ROOT,
    embed_batch_size=EMBED_BATCH_SIZE,
    run_gpt_enrichment=RUN_GPT_ENRICHMENT,
    max_stores=MAX_STORES,
)

print("Embedding deployment:", config.embedding_deployment)
print("GPT deployment:", config.gpt_deployment)
print("Search index:", config.search_index_name)
print("Max stores:", config.max_stores or "all")
print("GPT enrichment:", config.run_gpt_enrichment)

Embedding deployment: text-embedding-3-large
GPT deployment: gpt4-8k
Search index: store-dna-embeddings
Max stores: all
GPT enrichment: False


## 3. Load curated data (Stage 2 outputs)

In [8]:
tables = load_curated_tables(CURATED_DIR)

if MAX_STORES:
    store_ids = set(tables["dim_store"]["store_id"].head(MAX_STORES))
    for name in tables:
        if name == "dim_store":
            tables[name] = tables[name][tables[name]["store_id"].isin(store_ids)]
        elif "store_id" in tables[name].columns:
            tables[name] = tables[name][tables[name]["store_id"].isin(store_ids)]

for name, df in tables.items():
    print(f"{name:12} {len(df):>6,} rows")

dim_store       100 rows
reviews       1,318 rows
news          1,767 rows
reports         891 rows
products      1,630 rows
ops           5,200 rows


## 4. Preview embed text

Each row becomes a single string sent to the embedding model.

In [18]:
reports_preview = add_embed_text(tables["reports"], "reports")
print("Sample reportsembed_text:")
print(reports_preview["embed_text"].iloc[0][:300], "...")

Sample reportsembed_text:
Store: USR-001 | Report: RPT-USR-001-001 | Date: 2026-04-27 | Type: inventory_audit | Severity: medium | Category: inventory | Status: in_progress | Reported by: district_manager | Description: Cycle count variance exceeds threshold in high-shrink categories ...


## 5. Test embedding API (single batch)

Verify Azure credentials and embedding deployment before the full run.

In [10]:
client = config.create_client()

test_texts = reviews_preview["embed_text"].head(3).tolist()
test_vectors = embed_texts(
    client,
    test_texts,
    config.embedding_deployment,
    batch_size=3,
)

print("Test embedding shape:", test_vectors.shape)
print("First 5 dimensions:", test_vectors[0][:5])

Test embedding shape: (3, 3072)
First 5 dimensions: [-0.02903748 -0.00300026 -0.00292587  0.02281189  0.0145874 ]


## 6. Run full Stage 3 enrichment

Processes all modalities, uploads vectors to **Azure AI Search**, writes enriched CSVs locally.



In [11]:
summary = run_enrichment(CURATED_DIR, ENRICHED_DIR, config)
print(json.dumps(summary, indent=2))
print("Index document count:", get_index_document_count(config.search_config()))

[Stage 3] Loading curated tables...
[Stage 3] Processing all stores: 100
[Stage 3] Creating Azure OpenAI client...
[Stage 3] Ensuring Azure AI Search index: store-dna-embeddings
[Stage 3] reviews: 1,318 rows -> building embed_text
[Stage 3] reviews: embedding rows 1-50 of 1,318
[Stage 3] reviews: embedding rows 51-100 of 1,318
[Stage 3] reviews: embedding rows 101-150 of 1,318
[Stage 3] reviews: embedding rows 151-200 of 1,318
[Stage 3] reviews: embedding rows 201-250 of 1,318
[Stage 3] reviews: embedding rows 251-300 of 1,318
[Stage 3] reviews: embedding rows 301-350 of 1,318
[Stage 3] reviews: embedding rows 351-400 of 1,318
[Stage 3] reviews: embedding rows 401-450 of 1,318
[Stage 3] reviews: embedding rows 451-500 of 1,318
[Stage 3] reviews: embedding rows 501-550 of 1,318
[Stage 3] reviews: embedding rows 551-600 of 1,318
[Stage 3] reviews: embedding rows 601-650 of 1,318
[Stage 3] reviews: embedding rows 651-700 of 1,318
[Stage 3] reviews: embedding rows 701-750 of 1,318
[Stage 3

## 7. Inspect outputs

In [12]:
reviews_enriched = pd.read_csv(ENRICHED_DIR / "reviews_enriched.csv")
reviews_enriched.head(3)

,review_id,store_id,review_date,source,rating,sentiment,review_text,complaint_tags,embedding_model,embedding_dim
0,REV-USR-001-SITE-001,USR-001,2026-06-23,sitejabber,NaN,neutral,"Walmart's reputation is mixed, with customers ...",NaN,text-embedding-3-large,3072
1,REV-USR-001-SITE-002,USR-001,2026-06-23,sitejabber,NaN,positive,While some customers express loyalty and satis...,NaN,text-embedding-3-large,3072
2,REV-USR-001-SITE-003,USR-001,2026-06-23,sitejabber,NaN,negative,While Walmart ultimately provided refunds in s...,NaN,text-embedding-3-large,3072


In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

search_client = SearchClient(
    config.search_endpoint,
    config.search_index_name,
    AzureKeyCredential(config.search_api_key),
)
sample = search_client.search(
    search_text="*",
    filter="modality eq 'reviews'",
    select=["id", "store_id", "modality", "source_id"],
    top=5,
)
pd.DataFrame(list(sample))

,@search.score,id,store_id,modality,source_id,@search.reranker_score,@search.highlights,@search.captions,@search.document_debug_info,@search.reranker_boosted_score
0,1.0,reviews_REV-USR-001-SITE-006,USR-001,reviews,REV-USR-001-SITE-006,None,None,None,None,None
1,1.0,reviews_REV-USR-001-GOOG-002,USR-001,reviews,REV-USR-001-GOOG-002,None,None,None,None,None
2,1.0,reviews_REV-USR-002-SITE-008,USR-002,reviews,REV-USR-002-SITE-008,None,None,None,None,None
3,1.0,reviews_REV-USR-003-SITE-001,USR-003,reviews,REV-USR-003-SITE-001,None,None,None,None,None
4,1.0,reviews_REV-USR-006-SITE-007,USR-006,reviews,REV-USR-006-SITE-007,None,None,None,None,None


In [14]:
summary_path = ENRICHED_DIR / "store_modality_summaries.csv"
if summary_path.exists():
    pd.read_csv(summary_path).head(8)
else:
    print("No GPT summaries (RUN_GPT_ENRICHMENT was False)")

No GPT summaries (RUN_GPT_ENRICHMENT was False)


## 8. Output layout

```
data/USA_100_Stores/enriched/
├── reviews_enriched.csv
├── news_enriched.csv
├── reports_enriched.csv
├── products_enriched.csv
├── ops_weekly_enriched.csv
└── store_modality_summaries.csv   # GPT store × modality summaries

**Embeddings:** Azure AI Search index `AZURE_SEARCH_INDEX_NAME` (fields: `id`, `store_id`, `modality`, `source_id`, `content`, `content_vector`, …)
```

---

## 9. Next steps — Stage 4 (Modality Vectors)

| Action | Description |
|--------|-------------|
| Pool row embeddings | Query Azure AI Search by `store_id` + `modality`, mean-pool `content_vector` |
| Structured ops vector | Normalize KPIs from `fact_operations_weekly` |
| Output | One vector per modality per store → input to Stage 5 fusion |

